In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, count, col, length

In [0]:
df = spark.table("workspace.bronze.customers")
df.show()

In [0]:
total_rows = df.count()
unique_keys = df.select("customer_id").distinct().count()

print(f"Total Rows: {total_rows}")
print(f"Unique Order Keys: {unique_keys}")

if total_rows == unique_keys:
    print("✅ CHECK PASSED: customer_id is unique.")
else:
    print("❌ CHECK FAILED: There are duplicate order keys!")

In [0]:
df.groupBy("customer_unique_id") \
  .agg(count("customer_id").alias("order_count")) \
  .filter(col("order_count") > 1) \
  .orderBy(col("order_count").desc()) \
  .display()

The Zip-Code must contain 5 Digits. Upon Checking 0 is being removed from customer_zip_code_prefix by being stored as integers. Checking the number of incorrect zip codes

In [0]:
df.withColumn("zip_length", length(col("customer_zip_code_prefix").cast("string"))) \
  .filter(col("zip_length") < 5) \
  .select("customer_zip_code_prefix") \
  .distinct() \
  .display()

After Analysing I found out that -> 
customer_id is not a permanent ID. It is a unique key generated for every single order. If a customer buys something n times, they will have n different customer_id's.
customer_unique_id this is the permanent ID for the actual person.
So Changed the following name.

In [0]:
RENAME_MAP = {
    "customer_zip_code_prefix": "zip_prefix",
    "customer_id": "customer_order_key",
    "customer_unique_id": "customer_id"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
df.show()

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

In [0]:
df = df.filter(col("customer_id").isNotNull() & col("customer_order_key").isNotNull())

In [0]:
df = (
    df
    # State: Standardize to Uppercase
    .withColumn("customer_state", F.upper(F.substring(col("customer_state"), 1, 2)))
    
    # City: Lowercase and remove accents
    .withColumn("customer_city", F.lower(col("customer_city")))
    .withColumn("customer_city", F.regexp_replace(col("customer_city"), "[áàâãä]", "a"))
    .withColumn("customer_city", F.regexp_replace(col("customer_city"), "[éèêë]", "e"))
    .withColumn("customer_city", F.regexp_replace(col("customer_city"), "[íìîï]", "i"))
    .withColumn("customer_city", F.regexp_replace(col("customer_city"), "[óòôõö]", "o"))
    .withColumn("customer_city", F.regexp_replace(col("customer_city"), "[úùûü]", "u"))
    .withColumn("customer_city", F.regexp_replace(col("customer_city"), "[ç]", "c"))
    
    # Zip CodeCast to string first, then pad to 5 digits
    .withColumn("zip_prefix", F.lpad(col("zip_prefix").cast("string"), 5, "0"))
)

In [0]:
df = df.dropDuplicates(["customer_order_key"])

In [0]:
df = df.withColumn("silver_processed_time", F.current_timestamp())

In [0]:
df.printSchema()

In [0]:
df.display()

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.customers")

In [0]:
%sql
SELECT * FROM workspace.silver.customers